<img src="https://www.unad.edu.co/images/footer/logo-unad-acreditacion-min.png" width="780" height="140" align="right"/>

<p style="text-align: center;"> Curso: ENSEMBLE METHODS AND KERNELS</p>

<p style="text-align: center;"> Código Curso: 203008076 </p>

<p style="text-align: center;"> Grupo: 1 </p>

<p style="text-align: center;"> Phase 3 -Development of the Practical Component of the
Course Ensemble Methods and Kernels</p>

<p style="text-align: center;">  Presentado por: Wilmer Ricardo Urda</p>

<p style="text-align: center;"> Código: 1017194627</p>

<p style="text-align: center;">  Tutor: Ing. Jorge Luis Quintero Lopez </p>

<p style="text-align: center;"> UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA - UNAD </p>

# Exercise 4: Linear SVM, SVM with RBF Kernel, and the Kernel Trick

## SVM LINEAL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# CARGAR DATASET IRIS
# =========================
iris = load_iris()
X_full = pd.DataFrame(iris.data, columns=iris.feature_names)
y      = iris.target
class_names = list(iris.target_names)

# Features para visualización de límites de decisión (2D)
VIS_IDX   = [2, 3]  # petal length, petal width
X_vis     = X_full.iloc[:, VIS_IDX].values
vis_names = [iris.feature_names[i] for i in VIS_IDX]

# =========================
# SPLIT (todos los features para métricas)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# MODELO SVM LINEAL
# =========================
svm_linear = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='linear', C=1.0, random_state=42))
])

svm_linear.fit(X_train, y_train)
y_pred_lin = svm_linear.predict(X_test)

# =========================
# MÉTRICAS
# =========================
metrics_lin = pd.DataFrame([{
    'Model':     'SVM Linear',
    'Accuracy':  round(accuracy_score(y_test, y_pred_lin), 4),
    'Precision': round(precision_score(y_test, y_pred_lin, average='weighted'), 4),
    'Recall':    round(recall_score(y_test, y_pred_lin, average='weighted'), 4),
    'F1-score':  round(f1_score(y_test, y_pred_lin, average='weighted'), 4),
}])
print(metrics_lin.to_string(index=False))

In [ ]:
# =========================
# LEARNING CURVE – SVM LINEAL
# =========================
train_sizes_lin, train_scores_lin, test_scores_lin = learning_curve(
    svm_linear, X_full, y,
    cv=5,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes_lin, train_scores_lin.mean(axis=1), marker='o', label='Train Accuracy')
plt.plot(train_sizes_lin, test_scores_lin.mean(axis=1),  marker='o', label='Validation Accuracy')
plt.fill_between(train_sizes_lin,
                 train_scores_lin.mean(axis=1) - train_scores_lin.std(axis=1),
                 train_scores_lin.mean(axis=1) + train_scores_lin.std(axis=1), alpha=0.15)
plt.fill_between(train_sizes_lin,
                 test_scores_lin.mean(axis=1) - test_scores_lin.std(axis=1),
                 test_scores_lin.mean(axis=1) + test_scores_lin.std(axis=1), alpha=0.15)
plt.title('Learning Curve – SVM Linear (IRIS dataset)')
plt.xlabel('Training Size')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# LÍMITE DE DECISIÓN – SVM LINEAL (2 features)
# =========================
X_vis_train, X_vis_test, y_vis_train, y_vis_test = train_test_split(
    X_vis, y, test_size=0.2, random_state=42, stratify=y
)

svm_lin_vis = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='linear', C=1.0, random_state=42))
])
svm_lin_vis.fit(X_vis_train, y_vis_train)

h = 0.02
x_min, x_max = X_vis[:, 0].min() - 0.3, X_vis[:, 0].max() + 0.3
y_min, y_max = X_vis[:, 1].min() - 0.3, X_vis[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                      np.arange(y_min, y_max, h))
Z_lin = svm_lin_vis.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

colors_bg  = ['#FFD0D0', '#D0FFD0', '#D0D0FF']
colors_pts = ['#CC0000', '#008800', '#0000CC']

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z_lin, alpha=0.35, cmap=ListedColormap(colors_bg))
for cls, color in zip(range(3), colors_pts):
    mask = y == cls
    plt.scatter(X_vis[mask, 0], X_vis[mask, 1], c=color,
                label=class_names[cls], edgecolors='k', s=35, linewidths=0.5)
plt.title('Decision Boundary – SVM Linear\n(petal length vs petal width)')
plt.xlabel(vis_names[0])
plt.ylabel(vis_names[1])
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## SVM Linear – Analysis

The Support Vector Machine with a linear kernel was implemented using a `Pipeline` that first applies `StandardScaler` to normalize all features to zero mean and unit variance, then trains an `SVC(kernel='linear', C=1.0)`. Feature scaling is mandatory for SVMs because the margin optimization is distance-based and is directly affected by feature magnitudes.

**Mechanism:** The linear SVM finds the hyperplane that maximizes the margin between classes, defined by the support vectors (the training samples closest to the boundary). The decision function is `f(x) = w·x + b`, where `w` is the weight vector learned from the training data. With `C=1.0`, the regularization strikes a balance between maximizing the margin and minimizing misclassification.

**Learning curve:** The two curves (training and validation accuracy) show that:
- Training accuracy stabilizes quickly at high values as sample size grows.
- Validation accuracy converges toward training accuracy, showing good generalization.
- The narrow confidence bands confirm stable behavior across the five cross-validation folds.
- Minimal gap between the two curves indicates low variance and a well-calibrated regularization parameter.

**Decision boundary:** The linear kernel generates straight separating hyperplanes in the original feature space. The boundary between *setosa* and the other two classes is clearly linear. However, the boundary between *versicolor* and *virginica* is also linear, which may misclassify some samples in the overlapping region between these two species — a known limitation of the linear kernel on this dataset.

## SVM CON KERNEL RBF

In [ ]:
# =========================
# MODELO SVM RBF
# =========================
svm_rbf = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42))
])

svm_rbf.fit(X_train, y_train)
y_pred_rbf = svm_rbf.predict(X_test)

# =========================
# MÉTRICAS
# =========================
metrics_rbf = pd.DataFrame([{
    'Model':     'SVM RBF',
    'Accuracy':  round(accuracy_score(y_test, y_pred_rbf), 4),
    'Precision': round(precision_score(y_test, y_pred_rbf, average='weighted'), 4),
    'Recall':    round(recall_score(y_test, y_pred_rbf, average='weighted'), 4),
    'F1-score':  round(f1_score(y_test, y_pred_rbf, average='weighted'), 4),
}])
print(metrics_rbf.to_string(index=False))

In [ ]:
# =========================
# LEARNING CURVE – SVM RBF
# =========================
train_sizes_rbf, train_scores_rbf, test_scores_rbf = learning_curve(
    svm_rbf, X_full, y,
    cv=5,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes_rbf, train_scores_rbf.mean(axis=1), marker='o', label='Train Accuracy')
plt.plot(train_sizes_rbf, test_scores_rbf.mean(axis=1),  marker='o', label='Validation Accuracy')
plt.fill_between(train_sizes_rbf,
                 train_scores_rbf.mean(axis=1) - train_scores_rbf.std(axis=1),
                 train_scores_rbf.mean(axis=1) + train_scores_rbf.std(axis=1), alpha=0.15)
plt.fill_between(train_sizes_rbf,
                 test_scores_rbf.mean(axis=1) - test_scores_rbf.std(axis=1),
                 test_scores_rbf.mean(axis=1) + test_scores_rbf.std(axis=1), alpha=0.15)
plt.title('Learning Curve – SVM RBF (IRIS dataset)')
plt.xlabel('Training Size')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# LÍMITE DE DECISIÓN – SVM RBF (2 features)
# =========================
svm_rbf_vis = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42))
])
svm_rbf_vis.fit(X_vis_train, y_vis_train)

Z_rbf = svm_rbf_vis.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z_rbf, alpha=0.35, cmap=ListedColormap(colors_bg))
for cls, color in zip(range(3), colors_pts):
    mask = y == cls
    plt.scatter(X_vis[mask, 0], X_vis[mask, 1], c=color,
                label=class_names[cls], edgecolors='k', s=35, linewidths=0.5)
plt.title('Decision Boundary – SVM RBF\n(petal length vs petal width)')
plt.xlabel(vis_names[0])
plt.ylabel(vis_names[1])
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## SVM RBF – Analysis

The SVM with Radial Basis Function (Gaussian) kernel was implemented using the same `Pipeline` structure (`StandardScaler` + `SVC`), with `kernel='rbf'`, `C=1.0`, and `gamma='scale'`. The `gamma='scale'` setting automatically computes γ = 1 / (n_features × Var(X)), adapting the kernel width to the feature variance after scaling.

**Mechanism:** The RBF kernel implicitly maps each sample into an infinite-dimensional feature space using the transformation K(x, x') = exp(−γ‖x − x'‖²). Points near each other in the original space get a kernel value close to 1 (high similarity); distant points get values close to 0. The SVM then finds a linear separator in this implicit high-dimensional space, which corresponds to a non-linear boundary in the original space.

**Learning curve:** The two curves show that:
- Training accuracy reaches perfect (or near-perfect) values quickly and remains stable.
- Validation accuracy converges steadily to the training value as the dataset grows, confirming that the model generalizes well.
- The confidence bands are narrower than those of the linear model, indicating higher stability across folds.
- The early gap between training and validation accuracy is typical for non-linear kernels with small datasets, and closes as more samples are added.

**Decision boundary:** The RBF kernel produces curved, non-linear boundaries that can wrap more tightly around class clusters. The boundary between *versicolor* and *virginica* is smoother and better adapted to the actual data distribution than the linear boundary, potentially capturing more complex patterns in the overlap region.

## KERNEL TRICK

In [ ]:
# =========================
# KERNEL TRICK – VISUALIZACIÓN
# Demostración 1D → 2D: problema no separable linealmente
# se vuelve separable en el espacio de características phi(x) = (x, x²)
# =========================
np.random.seed(42)

# Clase A: valores extremos  |x| > 1
x_a = np.concatenate([np.random.uniform(-2.0, -1.0, 20),
                       np.random.uniform( 1.0,  2.0, 20)])
# Clase B: valores centrales |x| < 0.6
x_b = np.random.uniform(-0.6, 0.6, 20)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ---- Espacio original (1D) ----
ax0 = axes[0]
ax0.scatter(x_a, np.zeros_like(x_a), c='#CC0000', s=60,
            edgecolors='k', linewidths=0.5, label='Class A', zorder=5)
ax0.scatter(x_b, np.zeros_like(x_b), c='#0000CC', s=60,
            edgecolors='k', linewidths=0.5, label='Class B', zorder=5)
ax0.set_title('Original Space (1D)\nNot Linearly Separable', fontsize=11)
ax0.set_xlabel('x')
ax0.set_yticks([])
ax0.axhline(0, color='k', linewidth=0.5)
ax0.legend()
ax0.grid(True, alpha=0.3)

# ---- Espacio de características (2D) via phi(x) = (x, x²) ----
ax1 = axes[1]
ax1.scatter(x_a, x_a**2, c='#CC0000', s=60,
            edgecolors='k', linewidths=0.5, label='Class A', zorder=5)
ax1.scatter(x_b, x_b**2, c='#0000CC', s=60,
            edgecolors='k', linewidths=0.5, label='Class B', zorder=5)
ax1.axhline(0.45, color='green', linestyle='--', linewidth=2,
             label='Linear separator: x² = 0.45')
ax1.fill_between(np.linspace(-2.2, 2.2, 100), 0.45, 4.2,
                  color='#FFD0D0', alpha=0.25)
ax1.fill_between(np.linspace(-2.2, 2.2, 100), 0, 0.45,
                  color='#D0D0FF', alpha=0.25)
ax1.set_title('Feature Space (2D) via φ(x) = (x, x²)\nLinearly Separable', fontsize=11)
ax1.set_xlabel('x')
ax1.set_ylabel('x²')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

plt.suptitle('Kernel Trick: Mapping to a Higher-Dimensional Feature Space',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# COMPARACIÓN LADO A LADO: SVM LINEAL vs SVM RBF
# =========================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, Z, title in zip(axes,
                         [Z_lin, Z_rbf],
                         ['SVM Linear – Decision Boundary',
                          'SVM RBF – Decision Boundary']):
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=ListedColormap(colors_bg))
    for cls, color in zip(range(3), colors_pts):
        mask = y == cls
        ax.scatter(X_vis[mask, 0], X_vis[mask, 1], c=color,
                   label=class_names[cls], edgecolors='k', s=30, linewidths=0.5)
    ax.set_title(title)
    ax.set_xlabel(vis_names[0])
    ax.set_ylabel(vis_names[1])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Decision Boundary Comparison: Linear vs RBF Kernel\n(petal length vs petal width)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# =========================
# TABLA COMPARATIVA DE MÉTRICAS
# =========================
comparison = pd.concat([metrics_lin, metrics_rbf], ignore_index=True)
print('\nMetrics Comparison:')
print(comparison.to_string(index=False))

## Exercise 4 – Linear SVM, SVM with RBF Kernel, and the Kernel Trick

In this exercise, two SVM variants were implemented and compared on the **IRIS** dataset: a linear kernel SVM and an RBF (Gaussian) kernel SVM. Both models used a `Pipeline` with `StandardScaler` to prevent scale-dependent bias in the margin optimization.

---

### The Kernel Trick

The Kernel Trick is the central theoretical insight that allows SVMs to operate in high (or infinite) dimensional feature spaces without ever explicitly computing the feature map φ(x).

**Core idea:** SVM optimization depends only on dot products between training samples — specifically, terms of the form ⟨φ(xᵢ), φ(xⱼ)⟩. The kernel function K(xᵢ, xⱼ) = ⟨φ(xᵢ), φ(xⱼ)⟩ computes this dot product directly in the original space, without ever computing φ explicitly.

For the **RBF kernel**: K(x, x') = exp(−γ‖x − x'‖²)  
This corresponds to an *infinite-dimensional* feature map, making the separating hyperplane extremely flexible while maintaining computational tractability.

The visualization above illustrates this with a 1D example: points from two classes (A and B) that are not linearly separable in the original space become perfectly separable in the feature space φ(x) = (x, x²) using a simple horizontal hyperplane. The kernel trick lets the SVM exploit this separation without constructing the 2D mapping explicitly — it only needs the kernel value K(xᵢ, xⱼ) = xᵢ·xⱼ + (xᵢ·xⱼ)² (the polynomial kernel of degree 2).

---

### Performance Comparison

Both kernels achieve high classification accuracy on the IRIS dataset (a well-structured, low-noise benchmark). The RBF kernel typically matches or slightly exceeds the linear kernel's accuracy because:

- *Setosa* is perfectly linearly separable from the other two species, and both kernels capture this effortlessly.
- The boundary between *versicolor* and *virginica* has mild overlap. The RBF kernel's curved boundary can fit this region more precisely, reducing misclassifications on the overlapping samples.

The linear kernel performs strongly because the IRIS feature space is mostly linearly structured. Adding non-linear capacity (RBF) provides a marginal benefit but does not dramatically change results on this relatively simple dataset.

---

### Effect of the Kernel on Decision Boundaries

The side-by-side decision boundary plots (petal length vs petal width) reveal the fundamental geometric difference between the two kernels:

- **Linear kernel:** Decision boundaries are straight lines (hyperplanes in 2D). The class regions have flat, rigid edges. This is efficient and interpretable but cannot model curves or concavities in the true class distribution.
- **RBF kernel:** Decision boundaries are smooth curves that adapt to the local density of training samples. The boundary between *versicolor* and *virginica* bends around the actual data distribution, potentially reducing misclassifications in the overlap zone.

---

### Stability Analysis

Both learning curves show stable convergence:

- **Linear SVM:** Very narrow confidence bands across all training sizes. The model is robust to the choice of training subset, confirming that the linear structure of the data makes the boundary easy to learn reliably.
- **RBF SVM:** Slightly wider bands at small training sizes (fewer samples → more variability in the non-linear boundary), but these narrow quickly as training size grows. Both curves converge tightly by the full dataset size, confirming good generalization.

Neither model shows signs of overfitting: the training and validation accuracy curves converge rather than diverge as training size increases.

---

### Interpretability Discussion

- **Linear SVM:** Highly interpretable. The weight vector `w` directly indicates the importance of each feature in the decision. A positive weight for feature `i` means higher values push toward one class; negative means the opposite. This makes linear SVMs suitable for applications requiring regulatory transparency or feature attribution.
- **RBF SVM:** Much less interpretable. The decision function is a weighted sum of kernel evaluations over support vectors with no direct connection to individual features. Post-hoc methods (e.g., SHAP, LIME) are needed to explain predictions. This is a significant drawback in regulated domains.

---

### Methodological Justification

- **StandardScaler:** SVM maximizes a margin measured in Euclidean distance. Features with larger scales would otherwise dominate the margin calculation, making scaling mandatory — unlike tree-based methods.
- **C = 1.0:** A moderate regularization value that balances margin width against misclassification penalty. On IRIS, this default performs well without tuning.
- **gamma = 'scale':** Automatically sets γ = 1 / (n_features × Var(X_scaled)). Since X is already scaled, this produces γ = 1/n_features, a well-calibrated starting point that avoids extreme over- or under-smoothing.
- **Petal features for visualization:** Petal length and petal width provide the cleanest 2D separation of all three IRIS classes and are the most informative features for species classification, making them the best choice for decision boundary visualization.
- **stratify=y in split:** Ensures each class is proportionally represented in both train and test sets, critical for accurate metric estimation with a balanced multi-class dataset.

---

### Critical Analysis and Recommendations

**When to choose Linear SVM:**
- When the data is linearly separable or approximately so.
- When interpretability is required (feature weights are directly readable).
- When the dataset is large: linear SVMs scale much better than kernel SVMs (O(n) vs O(n²–n³)).
- In high-dimensional text classification (e.g., TF-IDF features), linear SVMs often outperform kernel methods.

**When to choose RBF SVM:**
- When the data has non-linear structure that a hyperplane cannot capture.
- When the dataset is small to medium (kernel computation becomes expensive for n > 10,000).
- Requires tuning of both C and γ — a grid search with cross-validation is strongly recommended.

**Limitation of both methods:**
SVM does not produce calibrated probability estimates by default. For tasks requiring class probabilities (e.g., risk scoring), `probability=True` must be set, which adds computational overhead via Platt scaling. For such scenarios, probabilistic classifiers (Logistic Regression, Random Forest) may be preferable.